# Day 034 — Exercise 5: AICli

**What you'll build:** The `AICli` class — `add_command(name, prompt_fn, help)` registers a subcommand that transforms `--input` text with `prompt_fn` before calling `ollama.chat`; `run(args_list)` parses and executes; `add_command` returns `self` for fluent chaining.

**Why it matters:** AICli is the full CLI tool in a class: any `prompt_fn` becomes a subcommand with one call to `add_command`.

## Provided: All Helper Functions

In [ ]:
from argparse import ArgumentParser

def make_parser() -> ArgumentParser:
    parser = ArgumentParser(
        prog='ai-tool',
        description='AI command-line tool powered by local LLM',
    )
    parser.add_argument(
        '--prompt', '-p', type=str, required=True,
        help='Prompt to send to the model',
    )
    parser.add_argument(
        '--model', '-m', type=str, default='llama3.2',
        help='Ollama model name (default: llama3.2)',
    )
    parser.add_argument(
        '--verbose', '-v', action='store_true',
        help='Print extra diagnostic output',
    )
    return parser


def make_extended_parser() -> ArgumentParser:
    parser = ArgumentParser(
        prog='ai-batch',
        description='AI batch processing tool',
    )
    parser.add_argument(
        '--prompt', '-p', type=str, required=True,
        help='Prompt text',
    )
    parser.add_argument(
        '--count', '-n', type=int, default=1,
        help='Number of completions (default: 1)',
    )
    parser.add_argument(
        '--format', '-f',
        choices=['text', 'json', 'markdown'],
        default='text',
        help='Output format (default: text)',
    )
    parser.add_argument(
        '--temperature', type=float, default=0.7,
        help='Sampling temperature 0.0-1.0 (default: 0.7)',
    )
    return parser


def make_subcommand_parser() -> ArgumentParser:
    parser = ArgumentParser(prog='ai-tool', description='AI CLI')
    subs   = parser.add_subparsers(dest='command', required=True,
                                   title='commands')

    chat = subs.add_parser('chat', help='Send a prompt to the AI')
    chat.add_argument('--prompt', '-p', required=True, help='The prompt')
    chat.add_argument('--model',  '-m', default='llama3.2')

    summarize = subs.add_parser('summarize', help='Summarize text')
    summarize.add_argument('--text',  '-t', required=True,
                           help='Text to summarize')
    summarize.add_argument('--model', '-m', default='llama3.2')

    return parser


def dispatch(ns, handlers: dict) -> str:
    cmd = ns.command
    if cmd not in handlers:
        raise KeyError(f"No handler registered for command: {cmd!r}")
    return handlers[cmd](ns)

## Your Implementation

In [ ]:
import ollama
from argparse import ArgumentParser

class AICli:
    """
    CLI wrapper: argparse subcommands + ollama LLM calls.
    Each subcommand maps --input text through a prompt_fn before calling
    ollama.chat. add_command returns self for fluent chaining.
    """

    def __init__(self, prog: str = 'ai-tool', model: str = 'llama3.2'):
        # TODO: self.model = model
        # TODO: self._parser = ArgumentParser(prog=prog, description='AI CLI')
        # TODO: self._subs = self._parser.add_subparsers(dest='command', required=True)
        # TODO: self._prompt_fns = {}
        pass

    def add_command(self, name: str, prompt_fn,
                    help: str = '') -> 'AICli':
        # TODO: sub = self._subs.add_parser(name, help=help)
        # TODO: sub.add_argument('--input', '-i', required=True, help='Input text')
        # TODO: self._prompt_fns[name] = prompt_fn
        # TODO: return self
        pass

    def run(self, args_list: list[str]) -> str:
        # TODO: ns = self._parser.parse_args(args_list)
        # TODO: prompt = self._prompt_fns[ns.command](ns.input)
        # TODO: resp = ollama.chat(
        #     model=self.model,
        #     messages=[{'role': 'user', 'content': prompt}],
        # )
        # TODO: return resp['message']['content']
        pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    # Check 1: class defined with add_command and run methods
    try:
        assert 'AICli' in globals()
        for m in ('add_command', 'run'):
            assert hasattr(AICli, m), f'missing method: {m}'
        passed += 1; print('\u2705 Check 1: AICli with add_command and run')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}')
        return

    # Check 2: __init__ stores model and creates parser
    try:
        cli = AICli(prog='test-tool', model='llama3.2')
        assert cli.model == 'llama3.2', \
            f'model: expected llama3.2, got {cli.model!r}'
        assert hasattr(cli, '_parser'), 'missing _parser attribute'
        assert hasattr(cli, '_prompt_fns'), 'missing _prompt_fns attribute'
        assert isinstance(cli._prompt_fns, dict), '_prompt_fns should be dict'
        passed += 1; print('\u2705 Check 2: __init__ stores model and creates parser')
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: add_command returns self (fluent) and registers prompt_fn
    try:
        cli = AICli()
        fn  = lambda t: f'test:{t}'
        ret = cli.add_command('ask', fn, help='Ask a question')
        assert ret is cli, f'add_command should return self, got {type(ret)}'
        assert 'ask' in cli._prompt_fns, \
            f'ask not registered: {list(cli._prompt_fns.keys())}'
        assert cli._prompt_fns['ask']('hello') == 'test:hello', \
            f'prompt_fn wrong: {cli._prompt_fns["ask"]("hello")!r}'
        passed += 1; print('\u2705 Check 3: add_command returns self and registers prompt_fn')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: run calls LLM and returns string
    try:
        cli = AICli(model='llama3.2')
        cli.add_command('ask', lambda t: t, help='Raw prompt')
        result = cli.run(['ask', '--input', 'Say the word hello'])
        assert isinstance(result, str), \
            f'run should return str, got {type(result).__name__}'
        assert result.strip(), 'run returned empty string'
        passed += 1; print(f'\u2705 Check 4: run returns non-empty string: {result.strip()[:40]!r}')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: fluent chaining and multiple commands
    try:
        cli = (
            AICli()
            .add_command('ask', lambda t: t)
            .add_command('upper', lambda t: f'UPPERCASE: {t}')
        )
        assert 'ask'   in cli._prompt_fns, 'ask not registered'
        assert 'upper' in cli._prompt_fns, 'upper not registered'
        assert len(cli._prompt_fns) >= 2, \
            f'expected >=2 commands, got {len(cli._prompt_fns)}'
        passed += 1; print('\u2705 Check 5: fluent chaining registers multiple commands')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
import ollama
from argparse import ArgumentParser

class AICli:
    def __init__(self, prog: str = 'ai-tool', model: str = 'llama3.2'):
        self.model       = model
        self._parser     = ArgumentParser(prog=prog,
                                          description='AI command-line tool')
        self._subs       = self._parser.add_subparsers(dest='command',
                                                        required=True)
        self._prompt_fns: dict = {}

    def add_command(self, name: str, prompt_fn,
                    help: str = '') -> 'AICli':
        sub = self._subs.add_parser(name, help=help)
        sub.add_argument('--input', '-i', required=True, help='Input text')
        self._prompt_fns[name] = prompt_fn
        return self

    def run(self, args_list: list[str]) -> str:
        ns     = self._parser.parse_args(args_list)
        prompt = self._prompt_fns[ns.command](ns.input)
        resp   = ollama.chat(
            model=self.model,
            messages=[{'role': 'user', 'content': prompt}],
        )
        return resp['message']['content']
```

</details>